# Unified RDF Converter Demo

This notebook demonstrates the new unified RDF conversion system for the AID-PAIS knowledge graph.

The system provides:
- **Modular architecture** supporting all 11+ adapters including UniChem
- **Concept-type specific handlers** for different biomedical entities
- **Adapter hints** for source-specific customizations
- **Integration** with CentralLookup for one-step search & convert
- **Batch processing** capabilities

## Features Demonstrated

- Converting UniChem results to RDF (✅ Working!)
- Using different concept type handlers
- Integrating with CentralLookup
- Merging with existing knowledge graphs
- Batch processing multiple queries

## Setup and Imports

In [27]:
# Import required libraries
from aid_pais_knowledgegraph.knowledge_lookup import CentralKnowledgeLookup
from aid_pais_knowledgegraph.knowledge_lookup.rdf_converter import (
    UnifiedRDFConverter,
    AdapterHints,
    RDFNamespaces
)
from aid_pais_knowledgegraph.knowledge_lookup.models import ConceptType, KnowledgeSource
import asyncio
import logging
import importlib

# Force reload the modules to pick up changes
importlib.reload(importlib.import_module('aid_pais_knowledgegraph.knowledge_lookup.models'))
importlib.reload(importlib.import_module('aid_pais_knowledgegraph.knowledge_lookup.rdf_converter'))

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("RDF conversion system loaded successfully!")
print(f"Supported concept types: {len(ConceptType)}")
print(f"Available knowledge sources: {len(KnowledgeSource)}")

RDF conversion system loaded successfully!
Supported concept types: 53
Available knowledge sources: 25


## Basic RDF Conversion

Let's start with basic RDF conversion using the UniChem adapter.

**Note**: The UniChem adapter searches by specific chemical identifiers (ChEMBL IDs, PubChem CIDs, etc.) rather than compound names. For example:
- Aspirin: `CHEMBL25`
- Caffeine: `CHEMBL113`
- Ibuprofen: `CHEMBL521`

In [20]:


lookup = CentralKnowledgeLookup()
print(f"Available adapters: {list(lookup.get_available_sources())}")

INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.umls_adapter:UMLS client initialized successfully
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized umls adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized bioportal adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized ols adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized wikidata adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized biolinker adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized dbpedia adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized oxo adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized umls adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized bioportal adapter
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Initialized ols adapter
INFO:aid_pais_kn

Available adapters: [<KnowledgeSource.UMLS: 'umls'>, <KnowledgeSource.BIOPORTAL: 'bioportal'>, <KnowledgeSource.OLS: 'ols'>, <KnowledgeSource.WIKIDATA: 'wikidata'>, <KnowledgeSource.BIOLINKER: 'biolinker'>, <KnowledgeSource.DBPEDIA: 'dbpedia'>, <KnowledgeSource.OXO: 'oxo'>, <KnowledgeSource.BIOONTOLOGY: 'bioontology'>, <KnowledgeSource.MONDO: 'mondo'>, <KnowledgeSource.UNIPROT: 'uniprot'>, <KnowledgeSource.UNICHEM: 'unichem'>]


In [21]:
# Search for aspirin using UniChem
print("\n=== Searching for Aspirin ===")
results = await lookup.search_concepts("CHEMBL25")  # Remove sources filter to use all available
print(f"Found {len(results.concepts)} concepts")
print(f"Sources queried: {[s.value for s in results.sources_queried]}")
print(f"Sources succeeded: {[s.value for s in results.sources_succeeded]}")
print(f"Sources failed: {[s.value for s in results.sources_failed]}")
print(f"Errors: {results.errors}")

# Display first concept
if results.concepts:
    concept = results.concepts[0]
    print(f"Primary ID: {concept.primary_id}")
    print(f"Label: {concept.primary_label}")
    print(f"Type: {concept.concept_type.value}")
    print(f"Identifiers: {len(concept.identifiers)}")
    print(f"Sources: {[s.value for s in concept.sources]}")
else:
    print("No concepts found.")


=== Searching for Aspirin ===


INFO:aid_pais_knowledgegraph.umls.auth:TGT refreshed
INFO:aid_pais_knowledgegraph.umls.search:Search for 'CHEMBL25' returned 0 results
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.umls_adapter:UMLS search for 'CHEMBL25' returned 0 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.biolinker_adapter:Calling BioLinker AI API for query: 'CHEMBL25' with search depth: 25
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.dbpedia_adapter:DBpedia API Request URL: https://dbpedia.org/sparql
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.dbpedia_adapter:DBpedia API Request Params: {'query': "\n            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n            PREFIX dbo: <http://dbpedia.org/ontology/>\n            SELECT DISTINCT ?resource ?label ?abstract ?type WHERE {\n              ?resource rdfs:label ?label .\n              FILTER (lang(?label) = 'en')\n              FILTER (regex(?label, 'CHEMBL25', 'i'))\n              OPTIONAL { ?resource db

Found 1 concepts
Sources queried: ['umls', 'bioportal', 'ols', 'wikidata', 'biolinker', 'dbpedia', 'oxo', 'bioontology', 'mondo', 'uniprot', 'unichem']
Sources succeeded: ['umls', 'bioportal', 'ols', 'wikidata', 'biolinker', 'dbpedia', 'oxo', 'bioontology', 'mondo', 'uniprot', 'unichem']
Sources failed: []
Errors: {}
Primary ID: 161671
Label: UCI_161671
Type: chemical
Identifiers: 52
Sources: ['unichem']


In [30]:
# Convert to RDF
print("\n=== Converting to RDF ===")
converter = UnifiedRDFConverter()
rdf_graph = converter.convert_concepts_to_graph(results.concepts)

print(f"RDF graph contains {len(rdf_graph)} triples")

# Show some sample triples
print("\nSample RDF triples:")
for i, (s, p, o) in enumerate(rdf_graph):
    if i >= 5:  # Show first 5 triples
        break
    print(f"  {s}")
    print(f"    {p}")
    print(f"    {o}")
    print()

INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Converted 1 concepts to RDF graph with 86 triples



=== Converting to RDF ===
RDF graph contains 86 triples

Sample RDF triples:
  http://www.aid-pais-kg.org/concept/161671
    http://www.aid-pais-kg.org/vocab/hasIdentifier
    unichem:acetylsalicylic acid

  http://www.aid-pais-kg.org/concept/161671
    http://www.aid-pais-kg.org/vocab/hasCategory
    fdasrs

  http://www.aid-pais-kg.org/concept/161671
    http://www.aid-pais-kg.org/vocab/hasIdentifier
    unichem:SCHEMBL1353

  http://www.aid-pais-kg.org/concept/161671
    http://www.aid-pais-kg.org/vocab/hasCategory
    brenda

  http://www.aid-pais-kg.org/concept/161671
    http://www.aid-pais-kg.org/vocab/conceptType
    chemical



In [5]:
# Save RDF to file
output_file = "./aspirin_rdf.ttl"
converter.save_graph(rdf_graph, output_file)
print(f"\nRDF saved to: {output_file}")

# Show file contents (first 20 lines)
print("\nFile contents preview:")
with open(output_file, 'r') as f:
    lines = f.readlines()[:20]
    print(''.join(lines))

INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Saved RDF graph with 86 triples to aspirin_rdf.ttl in turtle format



RDF saved to: ./aspirin_rdf.ttl

File contents preview:
@prefix aidpais: <http://www.aid-pais-kg.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix unichem: <https://www.ebi.ac.uk/unichem/compoundsources/> .
@prefix vocab: <http://www.aid-pais-kg.org/vocab/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

unichem:161671 a aidpais:Compound ;
    rdfs:label "UCI_161671" ;
    vocab:conceptType "chemical" ;
    vocab:confidenceScore "0.9"^^xsd:float ;
    vocab:hasCategory "CCDC",
        "MedChemExpress",
        "actor",
        "atlas",
        "bindingdb",
        "brenda",
        "chebi",
        "chembl",
        "chemicalbook",
        "clinicaltrials",



## One-Step Search and Convert

The CentralLookup now has a built-in method for search and RDF conversion in one step.

In [15]:
# One-step search and convert
print("=== One-Step Search & Convert ===")
# Note: UniChem searches by identifiers (CHEMBL, PubChem, etc.), not compound names
# For caffeine, use its ChEMBL ID: CHEMBL113
rdf_graph = await lookup.lookup_and_convert_to_rdf(
    query="CHEMBL113",  # Caffeine ChEMBL ID
    sources=[KnowledgeSource.UNICHEM],
    max_results=5,
    output_path="./caffeine_rdf.ttl"
)

print(f"Generated RDF graph with {len(rdf_graph)} triples")
print("RDF automatically saved to: ./caffeine_rdf.ttl")

=== One-Step Search & Convert ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL113' returned 1 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Search for 'CHEMBL113' completed in 1.78s. Found 1 concepts from 1 sources.
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Converted 1 concepts to RDF graph with 74 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Saved RDF graph with 74 triples to caffeine_rdf.ttl in turtle format
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:RDF results saved to: ./caffeine_rdf.ttl
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Converted 1 concepts to RDF graph with 74 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Search for 'CHEMBL113' completed in 1.78s. Found 1 concepts from 1 sources.
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Converted 1 concepts to RDF graph with 74 triples
INFO:aid_pais_knowledgegraph.knowledge_loo

Generated RDF graph with 74 triples
RDF automatically saved to: ./caffeine_rdf.ttl


## Concept Type Handlers

The system has specialized handlers for different concept types. Let's see how they work.

In [42]:
# Show supported concept types
print("=== Supported Concept Types ===")
supported_types = converter.get_supported_concept_types()
print(f"Number of handlers in converter: {len(converter.handlers)}")
print(f"Number of supported types returned: {len(supported_types)}")
for ct in sorted(supported_types, key=lambda x: x.value):
    print(f"- {ct.value}")

print(f"\nTotal supported types: {len(supported_types)}")

# Force reload the module
print("\n=== Forcing module reload ===")
import importlib
import sys

# Remove from sys.modules if it exists
modules_to_reload = [
    'aid_pais_knowledgegraph.knowledge_lookup.rdf_converter',
    'aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        print(f"Removing {module_name} from cache")
        del sys.modules[module_name]

# Re-import
try:
    from aid_pais_knowledgegraph.knowledge_lookup import rdf_converter
    importlib.reload(rdf_converter)
    print("Successfully reloaded rdf_converter module")
except Exception as e:
    print(f"Failed to reload: {e}")

# Test if from_ontology method exists
print("\n=== Testing from_ontology method ===")
try:
    from aid_pais_knowledgegraph.knowledge_lookup.rdf_converter import UnifiedRDFConverter
    print(f"Has from_ontology: {hasattr(UnifiedRDFConverter, 'from_ontology')}")
    if hasattr(UnifiedRDFConverter, 'from_ontology'):
        print("Method found!")
    else:
        print("Method still not found")
except Exception as e:
    print(f"Error checking method: {e}")

# Demonstrate dynamic loading
print("\n=== Dynamic Ontology Loading ===")
try:
    dynamic_converter = UnifiedRDFConverter.from_ontology()
    dynamic_types = dynamic_converter.get_supported_concept_types()
    print(f"Dynamic converter loaded {len(dynamic_types)} concept types from ontology")
    print("This automatically stays in sync with your AID-PAIS ontology!")
except Exception as e:
    print(f"Dynamic loading demo failed: {e}")

INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Loading 7 ontology files from /home/jhe24/AID-PAIS/AID-PAIS-KnowledgeGraph/hybrid_kg_prototype/ontology_mapping/aid-pais-ontology-modules/modules
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Loaded ontology graph with 615 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Extracted 49 concept classes from ontology
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Loaded 49 dynamic concept type handlers from ontology
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Loaded ontology graph with 615 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Extracted 49 concept classes from ontology
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Loaded 49 dynamic concept type handlers from ontology


=== Supported Concept Types ===
Number of handlers in converter: 53
Number of supported types returned: 53
- anatomical_entity
- anatomy
- assay
- biological_process
- biomarker
- case_control_study
- case_definition
- case_study
- cell_type
- cellular_component
- chemical
- citation
- clinical_study
- clinical_trial
- cohort_study
- community_trial
- cytokine
- demographic
- diagnostic_trial
- disease
- drug
- evidence
- gene
- gene_disease_association
- interventional_study
- laboratory_study
- meta_analysis
- metabolite
- molecular_entity
- molecular_function
- observation
- observational_study
- organ
- organ_system
- organism
- pathophysiological_process
- pathway
- person
- phenotype
- physiological_process
- procedure
- prognosis
- prospective_cohort_study
- protein
- randomized_controlled_trial
- reference
- retrospective_cohort_study
- study
- symptom
- systematic_review
- tissue
- treatment
- unknown

Total supported types: 53

=== Forcing module reload ===
Removing aid_pais_

In [40]:
# Demonstrate different handlers
print("\n=== Handler Examples ===")

# Chemical handler (UniChem)
from aid_pais_knowledgegraph.knowledge_lookup.rdf_converter import ChemicalHandler
chem_handler = ChemicalHandler(converter.namespaces)

# Create a mock chemical concept
from aid_pais_knowledgegraph.knowledge_lookup.models import UnifiedConcept, ConceptIdentifier
mock_chemical = UnifiedConcept(
    primary_id="161671",
    primary_label="UCI_161671",
    concept_type=ConceptType.CHEMICAL,
    identifiers=[
        ConceptIdentifier(KnowledgeSource.UNICHEM, "161671", "UniChem Compound"),
        ConceptIdentifier(KnowledgeSource.CHEMBL, "CHEMBL25", "Aspirin"),
    ]
)

print(f"Chemical concept URI: {chem_handler.get_primary_uri(mock_chemical)}")
print(f"Chemical class URI: {chem_handler.get_concept_class_uri(mock_chemical)}")


=== Handler Examples ===
Chemical concept URI: http://www.aid-pais-kg.org/compound/161671
Chemical class URI: http://www.aid-pais-kg.org/Compound


In [44]:
# Show dynamic concept types loaded from ontology
print("\n=== Dynamic Concept Types from Ontology ===")
try:
    dynamic_converter = UnifiedRDFConverter.from_ontology()
    dynamic_types = dynamic_converter.get_supported_concept_types()
    print(f"Dynamic converter loaded {len(dynamic_types)} concept types from ontology:")
    for ct in sorted(dynamic_types, key=lambda x: x.value):
        print(f"- {ct.value}")
    print(f"\nTotal dynamic types: {len(dynamic_types)}")
    print("These are automatically extracted from your AID-PAIS ontology owl:Class declarations!")
except Exception as e:
    print(f"Failed to show dynamic types: {e}")

print("\n=== Handler Examples ===")

# Chemical handler (UniChem)
from aid_pais_knowledgegraph.knowledge_lookup.rdf_converter import ChemicalHandler
chem_handler = ChemicalHandler(converter.namespaces)

# Create a mock chemical concept
from aid_pais_knowledgegraph.knowledge_lookup.models import UnifiedConcept, ConceptIdentifier
mock_chemical = UnifiedConcept(
    primary_id="161671",
    primary_label="UCI_161671",
    concept_type=ConceptType.CHEMICAL,
    identifiers=[
        ConceptIdentifier(KnowledgeSource.UNICHEM, "161671", "UniChem Compound"),
        ConceptIdentifier(KnowledgeSource.CHEMBL, "CHEMBL25", "Aspirin"),
    ]
)

print(f"Chemical concept URI: {chem_handler.get_primary_uri(mock_chemical)}")
print(f"Chemical class URI: {chem_handler.get_concept_class_uri(mock_chemical)}")

INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Loading 7 ontology files from /home/jhe24/AID-PAIS/AID-PAIS-KnowledgeGraph/hybrid_kg_prototype/ontology_mapping/aid-pais-ontology-modules/modules
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Loaded ontology graph with 663 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Extracted 49 concept classes from ontology
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Loaded 49 dynamic concept type handlers from ontology
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Loaded ontology graph with 663 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.ontology_concept_loader:Extracted 49 concept classes from ontology
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Loaded 49 dynamic concept type handlers from ontology



=== Dynamic Concept Types from Ontology ===
Dynamic converter loaded 49 concept types from ontology:
- anatomical_entity
- assay
- biological_process
- biomarker
- case_control_study
- case_definition
- case_study
- cell_type
- cellular_component
- citation
- clinical_study
- clinical_trial
- cohort_study
- community_trial
- cytokine
- demographic
- diagnostic_trial
- disease
- evidence
- gene
- gene_disease_association
- interventional_study
- laboratory_study
- meta_analysis
- metabolite
- molecular_entity
- molecular_function
- observation
- observational_study
- organ
- organ_system
- organism
- pathophysiological_process
- pathway
- person
- phenotype
- physiological_process
- prognosis
- prospective_cohort_study
- protein
- randomized_controlled_trial
- reference
- retrospective_cohort_study
- study
- symptom
- systematic_review
- tissue
- treatment
- unknown

Total dynamic types: 49
These are automatically extracted from your AID-PAIS ontology owl:Class declarations!

=== Handl

## Adapter Hints System

Adapter hints allow source-specific customizations of the RDF conversion.

In [9]:
# Create adapter hints for custom behavior
print("=== Adapter Hints Example ===")
hints = AdapterHints()

# Add custom namespace
from rdflib import Namespace
custom_ns = Namespace("https://example.org/custom/")
hints.add_namespace("custom", custom_ns)

# Add custom predicate
hints.add_predicate("hasCustomProperty", custom_ns.hasCustomProperty)

# Create converter with hints
converter_with_hints = UnifiedRDFConverter(hints)
print("Converter created with custom adapter hints")
print(f"Custom namespaces: {list(hints.custom_namespaces.keys())}")
print(f"Custom predicates: {list(hints.custom_predicates.keys())}")

=== Adapter Hints Example ===
Converter created with custom adapter hints
Custom namespaces: ['custom']
Custom predicates: ['hasCustomProperty']


## Merging with Existing Knowledge Graphs

The system can merge new data with existing knowledge graphs.

In [10]:
try:
    # Get some compound data - use ChEMBL ID for aspirin
    compound_results = await lookup.search_concepts(
        "CHEMBL25",  # Aspirin ChEMBL ID
        sources=[KnowledgeSource.UNICHEM],
        max_results=3
    )

=== Merging with Existing KG ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'aspirin' returned 0 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Search for 'aspirin' completed in 1.82s. Found 0 concepts from 1 sources.
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Search for 'aspirin' completed in 1.82s. Found 0 concepts from 1 sources.


No compound data found to merge


## Batch Processing

The system supports batch processing of multiple queries.

In [16]:
# Batch processing example
print("=== Batch Processing Example ===")

# Note: UniChem requires specific identifiers, not compound names
# Using ChEMBL IDs for the compounds
compound_ids = ["CHEMBL25", "CHEMBL113", "CHEMBL521"]  # Aspirin, Caffeine, Ibuprofen
batch_output = "./batch_compounds.ttl"

# Process first compound as example
batch_graph = await lookup.lookup_and_convert_to_rdf(
    query=compound_ids[0],  # Aspirin
    sources=[KnowledgeSource.UNICHEM],
    max_results=10,
    output_path=batch_output
)

print(f"Batch processing complete")
print(f"RDF graph contains {len(batch_graph)} triples")
print(f"Results saved to: {batch_output}")

# Note: For full batch processing with multiple queries, use the batch_rdf_converter.py script
print("\nFor processing multiple queries, use:")
print(f"python scripts/batch_rdf_converter.py --queries '{','.join(compound_ids)}' --output batch.ttl")

=== Batch Processing Example ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL25' returned 1 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Search for 'CHEMBL25' completed in 1.78s. Found 1 concepts from 1 sources.
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Converted 1 concepts to RDF graph with 86 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Saved RDF graph with 86 triples to batch_compounds.ttl in turtle format
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:RDF results saved to: ./batch_compounds.ttl
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Converted 1 concepts to RDF graph with 86 triples
INFO:aid_pais_knowledgegraph.knowledge_lookup.central_lookup:Search for 'CHEMBL25' completed in 1.78s. Found 1 concepts from 1 sources.
INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Converted 1 concepts to RDF graph with 86 triples
INFO:aid_pais_knowledgegraph.knowledge_

Batch processing complete
RDF graph contains 86 triples
Results saved to: ./batch_compounds.ttl

For processing multiple queries, use:
python scripts/batch_rdf_converter.py --queries 'CHEMBL25,CHEMBL113,CHEMBL521' --output batch.ttl


## Advanced Features

Let's explore some advanced features of the RDF conversion system.

In [12]:
# Advanced: Custom concept type handler
print("=== Custom Handler Example ===")

from aid_pais_knowledgegraph.knowledge_lookup.rdf_converter import ConceptTypeHandler

class CustomDrugHandler(ConceptTypeHandler):
    """Custom handler for drug concepts with additional properties."""

    def get_concept_class_uri(self, concept):
        return self.namespaces.AIDPAIS.Drug

    def get_primary_uri(self, concept):
        # Custom URI generation logic
        return self.namespaces.AIDPAIS[f"drug/{concept.primary_id}"]

    def add_type_specific_properties(self, graph, concept, concept_uri):
        # Add drug-specific properties
        for identifier in concept.identifiers:
            if identifier.source == KnowledgeSource.DRUGBANK:
                drugbank_uri = self.namespaces.DRUGBANK[identifier.identifier]
                graph.add((concept_uri, self.namespaces.VOCAB.hasDrugBankId, drugbank_uri))
                graph.add((drugbank_uri, self.namespaces.OWL.sameAs, concept_uri))

                # Add custom property
                graph.add((concept_uri, self.namespaces.VOCAB.hasCustomDrugProperty,
                          self.namespaces.RDF.Literal("Custom drug annotation")))

            # Add generic cross-reference
            graph.add((concept_uri, self.namespaces.VOCAB.hasIdentifier,
                      self.namespaces.RDF.Literal(f"{identifier.source.value}:{identifier.identifier}")))

# Register custom handler
custom_converter = UnifiedRDFConverter()
custom_handler = CustomDrugHandler(custom_converter.namespaces)
custom_converter.add_concept_type_handler(ConceptType.DRUG, custom_handler)

print("Custom drug handler registered")
print("This demonstrates how to extend the system for new concept types or custom behavior")

INFO:aid_pais_knowledgegraph.knowledge_lookup.rdf_converter:Registered custom handler for concept type: drug


=== Custom Handler Example ===
Custom drug handler registered
This demonstrates how to extend the system for new concept types or custom behavior


## Integration with Existing Workflows

The RDF converter integrates seamlessly with existing AID-PAIS workflows.

In [13]:
# Integration example
print("=== Integration with Existing Workflows ===")

# The system works with existing scripts and notebooks
print("Compatible with:")
print("- merge_ttls.py for combining RDF files")
print("- Existing knowledge graph validation")
print("- SPARQL query workflows")
print("- All existing AID-PAIS tooling")

# Show namespace alignment
print("\nNamespace alignment with existing KG:")
print(f"AIDPAIS base: {RDFNamespaces.AIDPAIS}")
print(f"VOCAB namespace: {RDFNamespaces.VOCAB}")
print("Matches existing SymptomGraph_v1.ttl structure")

# Demonstrate RDF format compatibility
print("\nSupported RDF formats:")
formats = ['turtle', 'ttl', 'xml', 'rdf', 'json-ld', 'nt', 'n3']
for fmt in formats:
    print(f"- {fmt}")

=== Integration with Existing Workflows ===
Compatible with:
- merge_ttls.py for combining RDF files
- Existing knowledge graph validation
- SPARQL query workflows
- All existing AID-PAIS tooling

Namespace alignment with existing KG:
AIDPAIS base: http://www.aid-pais-kg.org/
VOCAB namespace: http://www.aid-pais-kg.org/vocab/
Matches existing SymptomGraph_v1.ttl structure

Supported RDF formats:
- turtle
- ttl
- xml
- rdf
- json-ld
- nt
- n3


## Summary

This notebook demonstrated the new unified RDF conversion system:

### Key Features:
- **Modular Architecture**: Separate handlers for each concept type
- **Extensible**: Easy to add new concept types and adapters
- **Integrated**: Works seamlessly with CentralLookup
- **Compatible**: Aligns with existing AID-PAIS ontology
- **Batch Processing**: Supports large-scale conversion

### Usage Patterns:
1. **Simple conversion**: `converter.convert_concepts_to_graph(concepts)`
2. **One-step search & convert**: `lookup.lookup_and_convert_to_rdf(query, output_path)`
3. **Batch processing**: Use `batch_rdf_converter.py` script
4. **Custom handlers**: Extend for specific requirements

### Future Extensions:
- Add handlers for all 25+ adapters
- Implement adapter-specific hints
- Add validation and quality checks
- Integrate with reasoning and inference

The system is now ready for production use with UniChem and can be easily extended for all other knowledge sources!